# Historical pre-split grouping and manifest construction

Extracted from the original executed preparation cells. This is pre-split screening, not held-out model selection. The existing frozen manifest must not be replaced. No symbolic search is included.


In [ ]:
from pathlib import Path
import gc
import importlib.util
import itertools
import json
import math
import os
import platform
import re
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd

MPL_CACHE_DIR = Path("/tmp") / "matplotlib-nuclear-graphite-cache"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))
os.environ.setdefault("XDG_CACHE_HOME", str(MPL_CACHE_DIR))
os.environ.setdefault("MPLBACKEND", "Agg")

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception as exc:
    MATPLOTLIB_AVAILABLE = False
    plt = None
    print("matplotlib is unavailable; plot cells will be skipped.")
    print(exc)

try:
    import sympy as sp
    SYMPY_AVAILABLE = True
except Exception as exc:
    SYMPY_AVAILABLE = False
    sp = None
    print("sympy is unavailable; original-variable formula export will be skipped.")
    print(exc)

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = str

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 220)
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CASE_DIR = PROJECT_ROOT.parent / " Case Test Data" / "FE_Results_Cases_All"
if not CASE_DIR.exists():
    raise FileNotFoundError(f"Merged case directory does not exist: {CASE_DIR}")

SENSITIVITY_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "multi_case_sensitivity_199_cases"
MANIFEST_OVERRIDE = None
RESET_FROZEN_SPLIT = False

BASELINE_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "four_iteration_full_case_baseline_ml_199cases"
BASELINE_SPLIT_METRICS_PATH = BASELINE_OUTPUT_DIR / "baseline_split_metrics.csv"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "symbolic_regression_199_cases"
PYSR_RUNS_DIR = OUTPUT_DIR / "pysr_runs"
SIMILARITY_SCREEN_PATH = OUTPUT_DIR / "pre_split_case_similarity_pairs.csv"
SIMILARITY_GROUP_MAP_PATH = OUTPUT_DIR / "pre_split_similarity_group_map.csv"
FROZEN_MANIFEST_PATH = OUTPUT_DIR / "repeated_case_split_manifest_199cases.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PYSR_RUNS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
EXPECTED_CASE_COUNT = 199
EXPECTED_MISSING_CASE_NUMBERS = {25, 30}
EXPECTED_ELEMENTS_PER_CASE = 400_360
N_DEVELOPMENT_ITERATIONS = 4
FINAL_TEST_TARGET_CASES = 50
VALIDATION_TARGET_CASES = 15
INTERNAL_TEST_TARGET_CASES = 15
FULL_EVOLUTION_DATA_REQUIRED = True
SEARCH_ROWS_PER_TRAIN_CASE = EXPECTED_ELEMENTS_PER_CASE

# Conservative pre-split leakage guard. These values flag numerical similarity;
# they do not declare two FEM operating conditions physically identical.
SIMILARITY_DISTANCE_QUANTILE = 0.03
SIMILARITY_MUTUAL_NEIGHBOURS = 2
SIMILARITY_STRESS_P95_REL_TOL = 0.03
SIMILARITY_STRESS_P99_REL_TOL = 0.05
SIMILARITY_STRESS_MAX_REL_TOL = 0.10
FINAL_TEST_ANCHOR_METRICS = [
    "fluence_rate_mean",
    "temperature_mean",
    "weight_loss_rate_mean",
    "stress_p95",
    "stress_p99",
    "stress_max",
    "negative_stress_fraction",
]

MODEL_FEATURES = [
    "fluence_rate",
    "temperature",
    "weight_loss_rate",
    "rho",
    "theta",
    "z",
]
SCALED_FEATURE_NAMES = [f"{feature}_scaled" for feature in MODEL_FEATURES]
TARGET_COL = "sigma_max_principal"

CYLINDER_ORIGIN_X = 0.0
CYLINDER_ORIGIN_Y = 0.0
PROVIDED_THETA_UNIT = "radians"

# Start with a pilot. Unlock the final test only after the final formula is fixed.
# RUN_SYMBOLIC_SEARCH = False
RUN_SYMBOLIC_SEARCH = False
RUN_FINAL_TEST_EVALUATION = False
RUN_FORMULA_PASS_BENCHMARK = False
SEARCH_PROFILE = "four_iteration_timing_pilot"
OPERATOR_PROFILE = "extended_safe"
AUTO_INSTALL_PYSR_IF_MISSING = False
USE_BATCHING_DURING_EVOLUTION = False
BITWISE_DETERMINISTIC_SEARCH = False

if FULL_EVOLUTION_DATA_REQUIRED:
    if SEARCH_ROWS_PER_TRAIN_CASE != EXPECTED_ELEMENTS_PER_CASE:
        raise ValueError("Full evolution requires every element from every training case")
    if USE_BATCHING_DURING_EVOLUTION:
        raise ValueError("Full evolution requires USE_BATCHING_DURING_EVOLUTION=False")

print("Project root:", PROJECT_ROOT)
print("Case directory:", CASE_DIR)
print("Sensitivity evidence directory:", SENSITIVITY_OUTPUT_DIR)
print("Integrated similarity screen:", SIMILARITY_SCREEN_PATH)
print("Output directory:", OUTPUT_DIR)
print("Random seed:", RANDOM_SEED)
print("Model features:", MODEL_FEATURES)
print("Run symbolic search:", RUN_SYMBOLIC_SEARCH)
print("Run final test evaluation:", RUN_FINAL_TEST_EVALUATION)
print("Search profile:", SEARCH_PROFILE)
print("Full evolution data required:", FULL_EVOLUTION_DATA_REQUIRED)
print("PySR batching during evolution:", USE_BATCHING_DURING_EVOLUTION)
print("Rows per training case in equation evolution:", SEARCH_ROWS_PER_TRAIN_CASE)


## 3. Discover and reconcile the merged 199-case pool


In [ ]:
def extract_case_number(path: Path) -> int:
    match = re.search(r"Case_(\d+)", path.name, flags=re.IGNORECASE)
    if not match:
        raise ValueError(f"Cannot extract a case number from {path.name}")
    return int(match.group(1))


def count_data_rows(path: Path) -> int:
    newline_count = 0
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            newline_count += chunk.count(b"\n")
    return max(newline_count - 1, 0)


case_files = sorted(CASE_DIR.glob("FE_Results_Case_*.txt"), key=extract_case_number)
case_numbers = [extract_case_number(path) for path in case_files]
duplicate_case_numbers = sorted(
    number for number in set(case_numbers) if case_numbers.count(number) > 1
)
missing_case_numbers = sorted(set(range(0, 201)) - set(case_numbers))

if duplicate_case_numbers:
    raise ValueError(f"Duplicate case numbers: {duplicate_case_numbers}")
if len(case_files) != EXPECTED_CASE_COUNT:
    raise ValueError(f"Expected {EXPECTED_CASE_COUNT} case files, found {len(case_files)}")
if set(missing_case_numbers) != EXPECTED_MISSING_CASE_NUMBERS:
    raise ValueError(
        f"Expected missing cases {sorted(EXPECTED_MISSING_CASE_NUMBERS)}, "
        f"found {missing_case_numbers}"
    )

inventory_records = []
for path in case_files:
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        header = handle.readline().strip()
    inventory_records.append({
        "case_id": f"case_{extract_case_number(path):02d}",
        "case_number": extract_case_number(path),
        "file_name": path.name,
        "file_path": str(path),
        "size_mb": path.stat().st_size / 1_000_000,
        "n_elements_from_line_count": count_data_rows(path),
        "header": header,
    })

case_inventory = pd.DataFrame(inventory_records).sort_values("case_number").reset_index(drop=True)
unexpected_row_counts = case_inventory.loc[
    case_inventory["n_elements_from_line_count"] != EXPECTED_ELEMENTS_PER_CASE,
    ["case_id", "n_elements_from_line_count"],
]
if not unexpected_row_counts.empty:
    raise ValueError(
        "At least one case does not contain the expected 400,360 element rows:\n"
        + unexpected_row_counts.to_string(index=False)
    )
if case_inventory["header"].nunique() != 1:
    raise ValueError("The 199 case files do not share one identical header")

inventory_path = OUTPUT_DIR / "symbolic_case_file_inventory.csv"
case_inventory.to_csv(inventory_path, index=False)
print("Case files:", len(case_inventory))
print("Missing case numbers:", missing_case_numbers)
print("Total element rows:", int(case_inventory["n_elements_from_line_count"].sum()))
print("Saved:", inventory_path)
display(case_inventory.head())
display(case_inventory.tail())


## 4. Full-case reader and cylindrical preprocessing


In [ ]:
def normalise_column_name(name: object) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(name).strip().lower())


COLUMN_ALIASES = {
    "element_id": ["ElementID", "Element ID", "Element_Number", "ElementNumber"],
    "x": ["X", "X Coordinate", "X_Coordinate"],
    "y": ["Y", "Y Coordinate", "Y_Coordinate"],
    "z": ["Z", "Z Coordinate", "Z_Coordinate"],
    "rho": ["Rho", "Radius", "Radial Coordinate", "Radial_Coordinate"],
    "theta": ["Theta", "Theta Rad", "Theta_Rad", "Angular Coordinate", "Angular_Coordinate"],
    "fluence_rate": ["FluenceRate", "Fluence Rate", "Fast Neutron Fluence Rate"],
    "temperature": ["Temperature", "Temp"],
    "weight_loss_rate": ["WeightLossRate", "Weight Loss Rate", "Weight_Loss_Rate"],
    "sigma_max_principal": ["MaxPrincipalStress", "Max Principal Stress", "Maximum Principal Stress"],
}


def find_source_column(columns: list[str], standard_name: str) -> str | None:
    normalised_lookup = {}
    for column in columns:
        key = normalise_column_name(column)
        if key in normalised_lookup and normalised_lookup[key] != column:
            raise ValueError(
                f"Ambiguous columns after normalisation: {normalised_lookup[key]!r} and {column!r}"
            )
        normalised_lookup[key] = column
    for alias in COLUMN_ALIASES[standard_name]:
        source = normalised_lookup.get(normalise_column_name(alias))
        if source is not None:
            return source
    return None


def circular_angle_difference(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return np.abs(np.arctan2(np.sin(a - b), np.cos(a - b)))


def read_complete_case(path: Path) -> tuple[dict, dict]:
    raw = pd.read_csv(path, skipinitialspace=True)
    raw.columns = [str(column).strip() for column in raw.columns]
    case_number = extract_case_number(path)
    case_id = f"case_{case_number:02d}"

    source_columns = {
        name: find_source_column(list(raw.columns), name)
        for name in COLUMN_ALIASES
    }
    required = [
        "element_id", "z", "fluence_rate", "temperature",
        "weight_loss_rate", TARGET_COL,
    ]
    missing_required = [name for name in required if source_columns[name] is None]
    if missing_required:
        raise ValueError(f"{path.name} is missing required fields: {missing_required}")

    has_xy = source_columns["x"] is not None and source_columns["y"] is not None
    has_rho_theta = source_columns["rho"] is not None and source_columns["theta"] is not None
    if not has_xy and not has_rho_theta:
        raise ValueError(f"{path.name} must provide either X/Y or Rho/Theta")

    standard = pd.DataFrame(index=raw.index)
    for name in required:
        standard[name] = pd.to_numeric(raw[source_columns[name]], errors="coerce")

    rho_from_xy = None
    theta_from_xy = None
    if has_xy:
        x = pd.to_numeric(raw[source_columns["x"]], errors="coerce")
        y = pd.to_numeric(raw[source_columns["y"]], errors="coerce")
        x_relative = x - CYLINDER_ORIGIN_X
        y_relative = y - CYLINDER_ORIGIN_Y
        rho_from_xy = np.sqrt(x_relative ** 2 + y_relative ** 2)
        theta_from_xy = np.arctan2(y_relative, x_relative)

    if has_rho_theta:
        standard["rho"] = pd.to_numeric(raw[source_columns["rho"]], errors="coerce")
        theta = pd.to_numeric(raw[source_columns["theta"]], errors="coerce")
        if PROVIDED_THETA_UNIT == "degrees":
            theta = np.deg2rad(theta)
        elif PROVIDED_THETA_UNIT != "radians":
            raise ValueError("PROVIDED_THETA_UNIT must be 'radians' or 'degrees'")
        standard["theta"] = theta
        coordinate_source = "provided_rho_theta"
    else:
        standard["rho"] = rho_from_xy
        standard["theta"] = theta_from_xy
        coordinate_source = "derived_from_xy"

    expected_rows = int(
        case_inventory.loc[
            case_inventory["case_id"] == case_id,
            "n_elements_from_line_count",
        ].iloc[0]
    )
    if len(standard) != expected_rows:
        raise ValueError(
            f"{path.name}: pandas read {len(standard)} rows, expected {expected_rows}"
        )

    numeric_columns = ["element_id"] + MODEL_FEATURES + [TARGET_COL]
    missing_counts = standard[numeric_columns].isna().sum()
    infinite_counts = {
        column: int(np.isinf(standard[column].to_numpy(dtype=np.float64)).sum())
        for column in numeric_columns
    }
    if int(missing_counts.sum()) > 0:
        raise ValueError(
            f"{path.name} contains missing/non-numeric values: "
            f"{missing_counts[missing_counts > 0].to_dict()}"
        )
    if sum(infinite_counts.values()) > 0:
        raise ValueError(f"{path.name} contains infinite values: {infinite_counts}")

    element_values = standard["element_id"].to_numpy(dtype=np.float64)
    if not np.allclose(element_values, np.round(element_values), rtol=0.0, atol=0.0):
        raise ValueError(f"{path.name} contains non-integer ElementID values")
    element_ids = np.round(element_values).astype(np.int64)
    if np.unique(element_ids).size != len(element_ids):
        raise ValueError(f"{path.name} contains duplicate ElementID values")

    rho_crosscheck_max = np.nan
    theta_crosscheck_max = np.nan
    if has_xy and has_rho_theta:
        rho_crosscheck_max = float(
            np.max(np.abs(standard["rho"].to_numpy() - rho_from_xy.to_numpy()))
        )
        theta_crosscheck_max = float(
            np.max(
                circular_angle_difference(
                    standard["theta"].to_numpy(),
                    theta_from_xy.to_numpy(),
                )
            )
        )

    payload = {
        "X": np.ascontiguousarray(standard[MODEL_FEATURES].to_numpy(dtype=np.float32)),
        "y": np.ascontiguousarray(standard[TARGET_COL].to_numpy(dtype=np.float32)),
        "element_id": np.ascontiguousarray(element_ids),
    }
    audit = {
        "case_id": case_id,
        "case_number": case_number,
        "source_file": path.name,
        "n_elements_read": len(standard),
        "n_model_features": len(MODEL_FEATURES),
        "element_id_unique": int(np.unique(element_ids).size),
        "element_id_min": int(element_ids.min()),
        "element_id_max": int(element_ids.max()),
        "element_id_monotonic": bool(np.all(element_ids[1:] >= element_ids[:-1])),
        "coordinate_source": coordinate_source,
        "rho_xy_crosscheck_max_abs": rho_crosscheck_max,
        "theta_xy_crosscheck_max_abs_rad": theta_crosscheck_max,
        "model_missing_count": int(standard[MODEL_FEATURES + [TARGET_COL]].isna().sum().sum()),
        "model_infinite_count": int(sum(infinite_counts[column] for column in MODEL_FEATURES + [TARGET_COL])),
        "negative_stress_fraction": float((standard[TARGET_COL] < 0).mean()),
    }

    del raw, standard
    gc.collect()
    return payload, audit

In [ ]:
CASE_PATH_BY_ID = {
    row.case_id: Path(row.file_path)
    for row in case_inventory.itertuples(index=False)
}
CASE_NUMBER_BY_ID = {
    row.case_id: int(row.case_number)
    for row in case_inventory.itertuples(index=False)
}


def load_case_payload(case_id: str) -> tuple[dict, dict]:
    if case_id not in CASE_PATH_BY_ID:
        raise KeyError(f"Unknown case_id: {case_id}")
    return read_complete_case(CASE_PATH_BY_ID[case_id])


lazy_read_audit = pd.DataFrame({
    "case_id": list(CASE_PATH_BY_ID),
    "case_number": [CASE_NUMBER_BY_ID[case_id] for case_id in CASE_PATH_BY_ID],
    "source_file": [CASE_PATH_BY_ID[case_id].name for case_id in CASE_PATH_BY_ID],
    "n_elements_expected": EXPECTED_ELEMENTS_PER_CASE,
    "read_strategy": "lazy_full_case_read",
})
lazy_read_audit.to_csv(OUTPUT_DIR / "symbolic_lazy_read_plan.csv", index=False)

print("Cases registered for lazy full-case reading:", len(CASE_PATH_BY_ID))
print("The notebook will never keep all 199 complete element matrices in memory together.")
display(lazy_read_audit.head())


## 5. Pre-split similarity grouping and leakage guard

This step uses the completed 199-case sensitivity outputs rather than rescanning all 79,671,640 raw records. It compares:

1. Case-level distributions of the three physical predictor fields.
2. Stress distributions, including mean, p95, p99, maximum and negative-stress fraction.
3. Element-level correlations, high-stress feature contrasts and rho/theta/z hotspot profiles.

Case pairs enter a provisional similarity group only when they satisfy the three distance gates, mutual-neighbour criterion and the relative-difference gates for p95, p99 and maximum stress. Groups block closely related cases from crossing train, validation, internal-test and final-test splits. They do not establish that two FEM operating conditions are physically identical, and no case is deleted.

This records the historical pre-split construction, including its use of stress summaries. It must not be described as predictor-only grouping.


In [ ]:
SENSITIVITY_REQUIRED_FILES = {
    "case_summary": SENSITIVITY_OUTPUT_DIR / "case_summary.csv",
    "feature_correlations": SENSITIVITY_OUTPUT_DIR / "feature_correlations_by_case.csv",
    "high_stress_contrast": SENSITIVITY_OUTPUT_DIR / "high_stress_contrast_by_case.csv",
    "location_hotspots": SENSITIVITY_OUTPUT_DIR / "location_hotspot_fraction_by_case.csv",
}
missing_sensitivity_files = [
    str(path) for path in SENSITIVITY_REQUIRED_FILES.values() if not path.exists()
]
if missing_sensitivity_files:
    raise FileNotFoundError(
        "The 199-case sensitivity analysis must be completed before split construction. "
        f"Missing outputs: {missing_sensitivity_files}"
    )

similarity_case_summary = pd.read_csv(SENSITIVITY_REQUIRED_FILES["case_summary"])
similarity_correlations = pd.read_csv(SENSITIVITY_REQUIRED_FILES["feature_correlations"])
similarity_high_contrast = pd.read_csv(SENSITIVITY_REQUIRED_FILES["high_stress_contrast"])
similarity_hotspots = pd.read_csv(SENSITIVITY_REQUIRED_FILES["location_hotspots"])

expected_case_ids = set(CASE_PATH_BY_ID)
for table_name, table in {
    "case_summary": similarity_case_summary,
    "feature_correlations": similarity_correlations,
    "high_stress_contrast": similarity_high_contrast,
    "location_hotspots": similarity_hotspots,
}.items():
    observed_case_ids = set(table["case_id"].unique())
    if observed_case_ids != expected_case_ids:
        missing_ids = sorted(expected_case_ids - observed_case_ids)
        extra_ids = sorted(observed_case_ids - expected_case_ids)
        raise ValueError(
            f"{table_name} does not match the current 199-case pool; "
            f"missing={missing_ids}, extra={extra_ids}"
        )

ordered_case_ids = (
    similarity_case_summary.sort_values("case_number")["case_id"].tolist()
)


def standardize_similarity_block(frame: pd.DataFrame) -> tuple[np.ndarray, list[str]]:
    numeric = frame.reindex(ordered_case_ids).apply(pd.to_numeric, errors="coerce")
    usable_columns = [
        column
        for column in numeric.columns
        if numeric[column].notna().all() and numeric[column].std(ddof=0) > 0
    ]
    if not usable_columns:
        raise ValueError("A similarity feature block has no usable non-constant columns")
    values = numeric[usable_columns]
    standardized = (values - values.mean()) / values.std(ddof=0)
    if not np.isfinite(standardized.to_numpy(dtype=float)).all():
        raise ValueError("Non-finite values were produced while standardizing similarity features")
    return standardized.to_numpy(dtype=float), usable_columns


input_summary_columns = [
    f"{variable}_{statistic}"
    for variable in ["fluence_rate", "temperature", "weight_loss_rate"]
    for statistic in ["mean", "std", "min", "p95", "max"]
]
stress_summary_columns = [
    "stress_mean", "stress_std", "stress_p50", "stress_p95",
    "stress_p99", "stress_max", "negative_stress_fraction",
]

summary_indexed = similarity_case_summary.set_index("case_id")
input_matrix, input_similarity_columns = standardize_similarity_block(
    summary_indexed[input_summary_columns]
)
stress_matrix, stress_similarity_columns = standardize_similarity_block(
    summary_indexed[stress_summary_columns]
)

correlation_profile = (
    similarity_correlations[similarity_correlations["feature"].isin(MODEL_FEATURES)]
    .pivot(index="case_id", columns="feature", values="spearman")
    .add_prefix("spearman_")
)
contrast_profile = (
    similarity_high_contrast[similarity_high_contrast["feature"].isin(MODEL_FEATURES)]
    .pivot(index="case_id", columns="feature", values="standardized_difference")
    .add_prefix("high_stress_contrast_")
)
hotspot_profile = (
    similarity_hotspots[
        similarity_hotspots["feature"].isin(["rho_normalized", "theta", "z"])
    ]
    .pivot(index="case_id", columns=["feature", "bin"], values="high_stress_fraction")
)
hotspot_profile.columns = [
    f"hotspot_fraction_{feature}_bin_{int(bin_number):02d}"
    for feature, bin_number in hotspot_profile.columns
]
response_profile = correlation_profile.join(contrast_profile).join(hotspot_profile)
response_matrix, response_similarity_columns = standardize_similarity_block(response_profile)


def rms_distance(left: np.ndarray, right: np.ndarray) -> float:
    return float(np.sqrt(np.mean((left - right) ** 2)))


def symmetric_relative_difference(left: float, right: float) -> float:
    scale = max((abs(left) + abs(right)) / 2.0, 1e-12)
    return float(abs(left - right) / scale)


pair_records = []
for left_index, right_index in itertools.combinations(range(len(ordered_case_ids)), 2):
    case_a = ordered_case_ids[left_index]
    case_b = ordered_case_ids[right_index]
    summary_a = summary_indexed.loc[case_a]
    summary_b = summary_indexed.loc[case_b]
    pair_records.append({
        "case_a": case_a,
        "case_b": case_b,
        "case_number_a": CASE_NUMBER_BY_ID[case_a],
        "case_number_b": CASE_NUMBER_BY_ID[case_b],
        "input_distribution_distance": rms_distance(
            input_matrix[left_index], input_matrix[right_index]
        ),
        "stress_distribution_distance": rms_distance(
            stress_matrix[left_index], stress_matrix[right_index]
        ),
        "response_profile_distance": rms_distance(
            response_matrix[left_index], response_matrix[right_index]
        ),
        "stress_p95_relative_difference": symmetric_relative_difference(
            summary_a["stress_p95"], summary_b["stress_p95"]
        ),
        "stress_p99_relative_difference": symmetric_relative_difference(
            summary_a["stress_p99"], summary_b["stress_p99"]
        ),
        "stress_max_relative_difference": symmetric_relative_difference(
            summary_a["stress_max"], summary_b["stress_max"]
        ),
    })

similarity_pairs = pd.DataFrame(pair_records)
distance_columns = [
    "input_distribution_distance",
    "stress_distribution_distance",
    "response_profile_distance",
]
for column in distance_columns:
    similarity_pairs[f"{column}_percentile"] = similarity_pairs[column].rank(pct=True)
similarity_pairs["combined_distance_percentile_score"] = similarity_pairs[
    [f"{column}_percentile" for column in distance_columns]
].mean(axis=1)

neighbour_rank_lookup = {}
for case_id in ordered_case_ids:
    local_pairs = similarity_pairs[
        (similarity_pairs["case_a"] == case_id)
        | (similarity_pairs["case_b"] == case_id)
    ].copy()
    local_pairs["other_case"] = np.where(
        local_pairs["case_a"] == case_id,
        local_pairs["case_b"],
        local_pairs["case_a"],
    )
    local_pairs = local_pairs.sort_values(
        ["combined_distance_percentile_score", "other_case"]
    )
    neighbour_rank_lookup[case_id] = {
        other_case: rank
        for rank, other_case in enumerate(local_pairs["other_case"], start=1)
    }

similarity_pairs["rank_of_b_from_a"] = [
    neighbour_rank_lookup[case_a][case_b]
    for case_a, case_b in similarity_pairs[["case_a", "case_b"]].itertuples(index=False)
]
similarity_pairs["rank_of_a_from_b"] = [
    neighbour_rank_lookup[case_b][case_a]
    for case_a, case_b in similarity_pairs[["case_a", "case_b"]].itertuples(index=False)
]

distance_thresholds = {
    column: float(similarity_pairs[column].quantile(SIMILARITY_DISTANCE_QUANTILE))
    for column in distance_columns
}
similarity_pairs["similarity_candidate"] = (
    (similarity_pairs["rank_of_b_from_a"] <= SIMILARITY_MUTUAL_NEIGHBOURS)
    & (similarity_pairs["rank_of_a_from_b"] <= SIMILARITY_MUTUAL_NEIGHBOURS)
    & (similarity_pairs["input_distribution_distance"] <= distance_thresholds["input_distribution_distance"])
    & (similarity_pairs["stress_distribution_distance"] <= distance_thresholds["stress_distribution_distance"])
    & (similarity_pairs["response_profile_distance"] <= distance_thresholds["response_profile_distance"])
    & (similarity_pairs["stress_p95_relative_difference"] <= SIMILARITY_STRESS_P95_REL_TOL)
    & (similarity_pairs["stress_p99_relative_difference"] <= SIMILARITY_STRESS_P99_REL_TOL)
    & (similarity_pairs["stress_max_relative_difference"] <= SIMILARITY_STRESS_MAX_REL_TOL)
)
similarity_pairs["provisional_practical_screen"] = np.where(
    similarity_pairs["similarity_candidate"],
    "practically_close_under_integrated_presplit_screen",
    "difference_exceeds_integrated_presplit_screen",
)

similarity_pairs = similarity_pairs.sort_values(
    ["similarity_candidate", "combined_distance_percentile_score"],
    ascending=[False, True],
).reset_index(drop=True)
similarity_pairs.to_csv(SIMILARITY_SCREEN_PATH, index=False)
similarity_pairs.loc[similarity_pairs["similarity_candidate"]].to_csv(
    OUTPUT_DIR / "pre_split_similarity_candidate_pairs.csv", index=False
)

similarity_feature_inventory = pd.DataFrame([
    {"feature_block": "physical_input_distribution", "n_features": len(input_similarity_columns), "features": ";".join(input_similarity_columns)},
    {"feature_block": "stress_distribution", "n_features": len(stress_similarity_columns), "features": ";".join(stress_similarity_columns)},
    {"feature_block": "response_and_hotspot_profile", "n_features": len(response_similarity_columns), "features": ";".join(response_similarity_columns)},
])
similarity_feature_inventory.to_csv(
    OUTPUT_DIR / "pre_split_similarity_feature_inventory.csv", index=False
)
similarity_threshold_table = pd.DataFrame([
    {
        "distance_quantile": SIMILARITY_DISTANCE_QUANTILE,
        "mutual_neighbour_limit": SIMILARITY_MUTUAL_NEIGHBOURS,
        "input_distance_threshold": distance_thresholds["input_distribution_distance"],
        "stress_distance_threshold": distance_thresholds["stress_distribution_distance"],
        "response_profile_distance_threshold": distance_thresholds["response_profile_distance"],
        "stress_p95_relative_tolerance": SIMILARITY_STRESS_P95_REL_TOL,
        "stress_p99_relative_tolerance": SIMILARITY_STRESS_P99_REL_TOL,
        "stress_max_relative_tolerance": SIMILARITY_STRESS_MAX_REL_TOL,
        "n_all_case_pairs": len(similarity_pairs),
        "n_similarity_candidate_pairs": int(similarity_pairs["similarity_candidate"].sum()),
    }
])
similarity_threshold_table.to_csv(
    OUTPUT_DIR / "pre_split_similarity_thresholds.csv", index=False
)

# Build connected components so a chain of flagged neighbours cannot cross data splits.
parent = {case_id: case_id for case_id in ordered_case_ids}


def find_similarity_root(case_id: str) -> str:
    while parent[case_id] != case_id:
        parent[case_id] = parent[parent[case_id]]
        case_id = parent[case_id]
    return case_id


for row in similarity_pairs.loc[similarity_pairs["similarity_candidate"]].itertuples(index=False):
    root_a = find_similarity_root(row.case_a)
    root_b = find_similarity_root(row.case_b)
    if root_a != root_b:
        earlier, later = sorted([root_a, root_b], key=lambda case_id: CASE_NUMBER_BY_ID[case_id])
        parent[later] = earlier

roots = {case_id: find_similarity_root(case_id) for case_id in ordered_case_ids}
unique_roots = sorted(set(roots.values()), key=lambda case_id: CASE_NUMBER_BY_ID[case_id])
root_to_group = {root: f"group_{index:03d}" for index, root in enumerate(unique_roots)}
similarity_group_map = pd.DataFrame([
    {
        "case_id": case_id,
        "case_number": CASE_NUMBER_BY_ID[case_id],
        "similarity_group": root_to_group[roots[case_id]],
    }
    for case_id in ordered_case_ids
])
similarity_group_map.to_csv(SIMILARITY_GROUP_MAP_PATH, index=False)

similarity_group_summary = (
    similarity_group_map.groupby("similarity_group", as_index=False)
    .agg(
        group_size=("case_id", "size"),
        case_numbers=("case_number", lambda values: ";".join(map(str, sorted(values)))),
        case_ids=("case_id", lambda values: ";".join(values)),
    )
    .sort_values(["group_size", "similarity_group"], ascending=[False, True])
)
similarity_group_summary.to_csv(
    OUTPUT_DIR / "pre_split_similarity_group_summary.csv", index=False
)

print("All case pairs screened:", len(similarity_pairs))
print("Provisional similarity edges:", int(similarity_pairs["similarity_candidate"].sum()))
print("Similarity groups:", len(similarity_group_summary))
print("Non-singleton groups:", int((similarity_group_summary["group_size"] > 1).sum()))
print("Largest group size:", int(similarity_group_summary["group_size"].max()))
print("Saved:", SIMILARITY_SCREEN_PATH)
print("Saved:", SIMILARITY_GROUP_MAP_PATH)
display(similarity_threshold_table)
display(similarity_pairs.loc[similarity_pairs["similarity_candidate"]].head(20))
display(similarity_group_summary.head(20))

## 6. Freeze the group-aware case-level split manifest

The final-test allocation is constructed using a fixed random seed and boundary-coverage anchors, then read from the frozen manifest on later runs. In the historical construction code, a changed case pool or group map marks the manifest as stale and triggers rebuilding; otherwise the four rotations and final 50 cases remain fixed.

For this completed research archive, preserve the existing frozen manifest. Do not regenerate assignments in place or describe already evaluated final cases as an unopened test set. The delivery reproduction test runs this construction in temporary storage only.


In [ ]:
def build_similarity_groups(case_ids: list[str]) -> tuple[dict[str, str], str]:
    if not SIMILARITY_GROUP_MAP_PATH.exists():
        raise FileNotFoundError(
            "The integrated 199-case similarity-group map is missing. "
            "Run the preceding pre-split similarity cell before constructing the manifest."
        )
    group_map = pd.read_csv(SIMILARITY_GROUP_MAP_PATH)
    required = {"case_id", "case_number", "similarity_group"}
    if not required.issubset(group_map.columns):
        raise ValueError("The similarity-group map does not contain the required columns")
    if set(group_map["case_id"]) != set(case_ids):
        raise ValueError("The similarity-group map does not match the current case pool")
    if group_map["case_id"].duplicated().any():
        raise ValueError("At least one case appears more than once in the similarity-group map")
    return (
        group_map.set_index("case_id")["similarity_group"].to_dict(),
        "integrated_199case_presplit_similarity_groups",
    )


def group_members(
    group_by_case: dict[str, str],
    allowed_cases: set[str] | None = None,
) -> list[list[str]]:
    groups = {}
    for case_id, group_id in group_by_case.items():
        if allowed_cases is None or case_id in allowed_cases:
            groups.setdefault(group_id, []).append(case_id)
    return [
        sorted(members, key=lambda case_id: CASE_NUMBER_BY_ID[case_id])
        for _, members in sorted(groups.items())
    ]


def choose_grouped_cases(
    groups: list[list[str]],
    target_case_count: int,
    random_seed: int,
    required_case_ids: set[str] | None = None,
) -> tuple[set[str], list[list[str]]]:
    required_case_ids = set(required_case_ids or set())
    required_groups = [
        members for members in groups if required_case_ids.intersection(members)
    ]
    optional_groups = [members for members in groups if members not in required_groups]
    selected = list(required_groups)
    selected_count = sum(len(members) for members in selected)
    if selected_count > target_case_count:
        raise ValueError(
            f"Required coverage groups contain {selected_count} cases, "
            f"which exceeds the target {target_case_count}"
        )

    rng = np.random.default_rng(random_seed)
    shuffled = [optional_groups[index] for index in rng.permutation(len(optional_groups))]
    remaining = []
    for members in shuffled:
        if selected_count + len(members) <= target_case_count:
            selected.append(members)
            selected_count += len(members)
        else:
            remaining.append(members)

    if selected_count < target_case_count and remaining:
        best_index = min(
            range(len(remaining)),
            key=lambda index: abs(target_case_count - (selected_count + len(remaining[index]))),
        )
        candidate = remaining[best_index]
        if abs(target_case_count - (selected_count + len(candidate))) < abs(
            target_case_count - selected_count
        ):
            selected.append(candidate)
            selected_count += len(candidate)
            remaining.pop(best_index)

    selected_cases = {case_id for members in selected for case_id in members}
    return selected_cases, remaining


def determine_final_test_anchor_cases() -> tuple[set[str], pd.DataFrame]:
    summary = similarity_case_summary.set_index("case_id")
    records = []
    for metric in FINAL_TEST_ANCHOR_METRICS:
        if metric not in summary.columns:
            raise ValueError(f"Final-test anchor metric is unavailable: {metric}")
        values = pd.to_numeric(summary[metric], errors="coerce")
        if values.isna().any():
            raise ValueError(f"Final-test anchor metric contains missing values: {metric}")
        for boundary, case_id in [("minimum", values.idxmin()), ("maximum", values.idxmax())]:
            records.append({
                "metric": metric,
                "boundary": boundary,
                "case_id": case_id,
                "case_number": CASE_NUMBER_BY_ID[case_id],
                "value": float(values.loc[case_id]),
            })
    audit = pd.DataFrame(records)
    return set(audit["case_id"]), audit


def generate_split_manifest() -> tuple[pd.DataFrame, str, pd.DataFrame]:
    case_ids = sorted(CASE_PATH_BY_ID, key=lambda case_id: CASE_NUMBER_BY_ID[case_id])
    group_by_case, grouping_source = build_similarity_groups(case_ids)
    all_groups = group_members(group_by_case)
    anchor_cases, anchor_audit = determine_final_test_anchor_cases()
    final_cases, _ = choose_grouped_cases(
        all_groups,
        FINAL_TEST_TARGET_CASES,
        RANDOM_SEED,
        required_case_ids=anchor_cases,
    )
    development_cases = set(case_ids) - final_cases

    records = []
    for iteration in range(1, N_DEVELOPMENT_ITERATIONS + 1):
        development_groups = group_members(group_by_case, development_cases)
        validation_cases, remaining_groups = choose_grouped_cases(
            development_groups,
            VALIDATION_TARGET_CASES,
            RANDOM_SEED + 100 * iteration,
        )
        remaining_after_validation = [
            members
            for members in remaining_groups
            if not any(case_id in validation_cases for case_id in members)
        ]
        internal_test_cases, _ = choose_grouped_cases(
            remaining_after_validation,
            INTERNAL_TEST_TARGET_CASES,
            RANDOM_SEED + 100 * iteration + 1,
        )

        for case_id in case_ids:
            if case_id in final_cases:
                split = "final_test"
            elif case_id in validation_cases:
                split = "validation"
            elif case_id in internal_test_cases:
                split = "internal_test"
            else:
                split = "train"
            records.append({
                "iteration": iteration,
                "case_id": case_id,
                "case_number": CASE_NUMBER_BY_ID[case_id],
                "similarity_group": group_by_case[case_id],
                "split": split,
            })
    anchor_audit["required_split"] = "final_test"
    return pd.DataFrame(records), grouping_source, anchor_audit


required_manifest_columns = {
    "iteration", "case_id", "case_number", "similarity_group", "split"
}


def validate_manifest_against_current_groups(
    candidate_manifest: pd.DataFrame,
    expected_group_by_case: dict[str, str],
) -> tuple[bool, str]:
    if not required_manifest_columns.issubset(candidate_manifest.columns):
        return False, "required columns are missing"
    if set(candidate_manifest["iteration"].unique()) != set(
        range(1, N_DEVELOPMENT_ITERATIONS + 1)
    ):
        return False, "iteration IDs do not match the four configured iterations"
    if set(candidate_manifest["case_id"].unique()) != set(CASE_PATH_BY_ID):
        return False, "case IDs do not match the current 199-case pool"
    observed_groups = (
        candidate_manifest[["case_id", "similarity_group"]]
        .drop_duplicates()
    )
    if observed_groups["case_id"].duplicated().any():
        return False, "a case changes similarity group between iterations"
    observed_group_by_case = observed_groups.set_index("case_id")["similarity_group"].to_dict()
    if observed_group_by_case != expected_group_by_case:
        return False, "similarity groups differ from the current integrated screen"

    for iteration, rows in candidate_manifest.groupby("iteration"):
        if len(rows) != EXPECTED_CASE_COUNT or rows["case_id"].nunique() != EXPECTED_CASE_COUNT:
            return False, f"iteration {iteration} does not contain each case exactly once"
        leakage = rows.groupby("similarity_group")["split"].nunique()
        if (leakage > 1).any():
            return False, f"iteration {iteration} splits a similarity group"
        counts = rows["split"].value_counts()
        expected_counts = {
            "train": EXPECTED_CASE_COUNT - FINAL_TEST_TARGET_CASES - VALIDATION_TARGET_CASES - INTERNAL_TEST_TARGET_CASES,
            "validation": VALIDATION_TARGET_CASES,
            "internal_test": INTERNAL_TEST_TARGET_CASES,
            "final_test": FINAL_TEST_TARGET_CASES,
        }
        if counts.to_dict() != expected_counts:
            return False, f"iteration {iteration} has unexpected split counts: {counts.to_dict()}"

    final_sets = [
        frozenset(rows.loc[rows["split"] == "final_test", "case_id"])
        for _, rows in candidate_manifest.groupby("iteration")
    ]
    if len(set(final_sets)) != 1:
        return False, "the final-test set changes between iterations"
    return True, "valid frozen group-aware manifest"


case_ids_for_manifest = sorted(CASE_PATH_BY_ID, key=lambda case_id: CASE_NUMBER_BY_ID[case_id])
current_group_by_case, current_grouping_source = build_similarity_groups(case_ids_for_manifest)
manifest_was_reused = False
manifest_rebuild_reason = "no frozen manifest existed"
final_test_anchor_audit = pd.DataFrame()

if MANIFEST_OVERRIDE is not None:
    split_manifest = pd.read_csv(MANIFEST_OVERRIDE)
    manifest_source = f"user_manifest:{MANIFEST_OVERRIDE}"
    is_valid, validation_message = validate_manifest_against_current_groups(
        split_manifest, current_group_by_case
    )
    if not is_valid:
        raise ValueError(f"MANIFEST_OVERRIDE is invalid: {validation_message}")
elif FROZEN_MANIFEST_PATH.exists() and not RESET_FROZEN_SPLIT:
    frozen_candidate = pd.read_csv(FROZEN_MANIFEST_PATH)
    is_valid, validation_message = validate_manifest_against_current_groups(
        frozen_candidate, current_group_by_case
    )
    if is_valid:
        split_manifest = frozen_candidate
        manifest_source = f"frozen_manifest:{FROZEN_MANIFEST_PATH.name}"
        manifest_was_reused = True
        manifest_rebuild_reason = "not rebuilt; existing manifest passed all safeguards"
    else:
        superseded_path = OUTPUT_DIR / "repeated_case_split_manifest_199cases_superseded.csv"
        frozen_candidate.to_csv(superseded_path, index=False)
        split_manifest, grouping_source, final_test_anchor_audit = generate_split_manifest()
        manifest_source = f"regenerated_after_stale_manifest:{validation_message}"
        manifest_rebuild_reason = validation_message
        print("Previous manifest was superseded:", validation_message)
        print("Saved previous manifest:", superseded_path)
else:
    split_manifest, grouping_source, final_test_anchor_audit = generate_split_manifest()
    manifest_source = grouping_source
    manifest_rebuild_reason = (
        "RESET_FROZEN_SPLIT=True" if RESET_FROZEN_SPLIT else "no frozen manifest existed"
    )

is_valid, validation_message = validate_manifest_against_current_groups(
    split_manifest, current_group_by_case
)
if not is_valid:
    raise AssertionError(f"Generated split manifest failed validation: {validation_message}")

if final_test_anchor_audit.empty:
    _, final_test_anchor_audit = determine_final_test_anchor_cases()
    final_test_anchor_audit["required_split"] = "final_test"
final_test_cases = set(
    split_manifest.loc[
        (split_manifest["iteration"] == 1)
        & (split_manifest["split"] == "final_test"),
        "case_id",
    ]
)
final_test_anchor_audit["included_in_frozen_final_test"] = (
    final_test_anchor_audit["case_id"].isin(final_test_cases)
)
if not final_test_anchor_audit["included_in_frozen_final_test"].all():
    raise AssertionError("At least one configured boundary anchor is absent from final test")

split_manifest.to_csv(FROZEN_MANIFEST_PATH, index=False)
final_test_anchor_audit.to_csv(
    OUTPUT_DIR / "final_test_boundary_anchor_audit.csv", index=False
)

split_counts = split_manifest.groupby(["iteration", "split"]).size().unstack(fill_value=0)
group_leakage_audit = (
    split_manifest.groupby(["iteration", "similarity_group"])["split"]
    .nunique()
    .rename("n_splits_used_by_group")
    .reset_index()
)
group_leakage_audit["leakage_detected"] = (
    group_leakage_audit["n_splits_used_by_group"] > 1
)
group_leakage_audit.to_csv(
    OUTPUT_DIR / "similarity_group_cross_split_leakage_audit.csv", index=False
)

coverage_records = []
coverage_summary = similarity_case_summary.set_index("case_id")
for iteration, iteration_rows in split_manifest.groupby("iteration"):
    for split_name, split_rows in iteration_rows.groupby("split"):
        split_case_ids = split_rows["case_id"].tolist()
        for metric in FINAL_TEST_ANCHOR_METRICS:
            pool_values = pd.to_numeric(coverage_summary[metric], errors="coerce")
            split_values = pool_values.reindex(split_case_ids)
            coverage_records.append({
                "iteration": iteration,
                "split": split_name,
                "metric": metric,
                "n_cases": len(split_values),
                "split_min": float(split_values.min()),
                "split_mean": float(split_values.mean()),
                "split_max": float(split_values.max()),
                "pool_min": float(pool_values.min()),
                "pool_mean": float(pool_values.mean()),
                "pool_max": float(pool_values.max()),
                "n_in_pool_bottom_10pct": int((split_values <= pool_values.quantile(0.10)).sum()),
                "n_in_pool_top_10pct": int((split_values >= pool_values.quantile(0.90)).sum()),
            })
split_coverage_audit = pd.DataFrame(coverage_records)
split_coverage_audit.to_csv(
    OUTPUT_DIR / "split_boundary_coverage_audit.csv", index=False
)

freeze_metadata = {
    "case_count": EXPECTED_CASE_COUNT,
    "random_seed": RANDOM_SEED,
    "development_iterations": N_DEVELOPMENT_ITERATIONS,
    "final_test_target_cases": FINAL_TEST_TARGET_CASES,
    "validation_target_cases": VALIDATION_TARGET_CASES,
    "internal_test_target_cases": INTERNAL_TEST_TARGET_CASES,
    "similarity_grouping_source": current_grouping_source,
    "manifest_source": manifest_source,
    "manifest_was_reused": manifest_was_reused,
    "manifest_rebuild_reason": manifest_rebuild_reason,
    "similarity_distance_quantile": SIMILARITY_DISTANCE_QUANTILE,
    "similarity_mutual_neighbours": SIMILARITY_MUTUAL_NEIGHBOURS,
    "stress_p95_relative_tolerance": SIMILARITY_STRESS_P95_REL_TOL,
    "stress_p99_relative_tolerance": SIMILARITY_STRESS_P99_REL_TOL,
    "stress_max_relative_tolerance": SIMILARITY_STRESS_MAX_REL_TOL,
    "final_test_case_ids": sorted(final_test_cases, key=lambda case_id: CASE_NUMBER_BY_ID[case_id]),
}
(OUTPUT_DIR / "frozen_split_metadata.json").write_text(
    json.dumps(freeze_metadata, indent=2), encoding="utf-8"
)

print("Manifest source:", manifest_source)
print("Manifest validation:", validation_message)
print("Frozen final-test cases:", len(final_test_cases))
print("Similarity-group cross-split leakage rows:", int(group_leakage_audit["leakage_detected"].sum()))
print("Saved:", FROZEN_MANIFEST_PATH)
display(split_counts.reindex(columns=["train", "validation", "internal_test", "final_test"]))
display(final_test_anchor_audit)
display(split_manifest.head())